# Emotion Detection with a CNN -- FER2013

**Goal**: classify a 48x48 grayscale face crop into one of 7 emotions -- angry, disgust, fear,
happy, neutral, sad, surprise -- using a CNN, the same convolution-based approach from the
Fashion-MNIST notebook, but on a genuinely harder, real-world dataset.

**Dataset**: [FER2013](https://www.kaggle.com/datasets/msambare/fer2013) -- 35,887 grayscale
48x48 images, split into `train/` (28,709 images) and `test/` (7,178 images), each already
organized into 7 class subfolders. It was originally released for a 2013 Kaggle/ICML challenge.

**Why this is harder than Fashion-MNIST**:
- Faces are *real photos* scraped from the web -- varied lighting, angle, occlusion (glasses,
  hands), and age/ethnicity, not clean studio garment photos.
- The classes are **imbalanced** -- `happy` has 7,215 training images but `disgust` has only
  436. A model can get "decent" overall accuracy while being nearly blind to `disgust`.
- Emotion labels are inherently subjective -- even human annotators agree on FER2013 labels only
  about 65-70% of the time. That human-agreement number is a realistic upper bound to compare
  our model's accuracy against -- getting close to it is a success, not a failure to hit 95%+.

In [ ]:
import os
import pathlib

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

DATA_DIR = pathlib.Path("fer2013")
IMG_SIZE = (48, 48)
BATCH_SIZE = 64
SEED = 42

## 1. Load the data

`keras.utils.image_dataset_from_directory` reads images straight from the class-per-folder
layout Kaggle already gives us (`train/angry/*.jpg`, `train/happy/*.jpg`, ...) and infers the
label from the folder name -- no manual label wrangling needed, unlike the flat `fashion_mnist`
arrays from the previous notebook.

The dataset only ships `train/` and `test/`, so we carve 10% off `train/` for validation
(`validation_split` + matching `seed` on both calls keeps the split identical and non-overlapping).

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    labels="inferred",
    label_mode="int",
    color_mode="grayscale",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    subset="training",
    seed=SEED,
)

val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    labels="inferred",
    label_mode="int",
    color_mode="grayscale",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    subset="validation",
    seed=SEED,
)

test_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "test",
    labels="inferred",
    label_mode="int",
    color_mode="grayscale",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = train_ds.class_names
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(2000, seed=SEED).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

## 2. Look at the class imbalance

Before building anything, look at how lopsided the training set is. This number directly
explains a design choice later (`class_weight` in `model.fit`).

In [ ]:
train_dir = DATA_DIR / "train"
class_counts = {c: len(os.listdir(train_dir / c)) for c in class_names}

plt.figure(figsize=(7, 4))
plt.bar(class_counts.keys(), class_counts.values(), color="#4C72B0")
plt.title("Training images per class (FER2013)")
plt.ylabel("# images")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(class_counts)
print(f"\nLargest class ('happy') is {max(class_counts.values()) / min(class_counts.values()):.1f}x "
      f"bigger than the smallest class ('disgust').")

## 3. Build the model

Same core recipe as the Fashion-MNIST CNN (`Conv2D -> MaxPooling2D`, stacked, then `Flatten ->
Dense -> softmax`), with three additions justified by this being a harder problem:

- **A third conv block** (32 -> 64 -> 128 filters). 48x48 faces have more real structure to
  learn than 28x28 clothing silhouettes, so we give the network more capacity -- this is the
  same design tradeoff explored in the Day 3 TODO 1 exercise, except here the extra depth is
  actually justified by the harder task.
- **BatchNormalization after every conv layer**. It rescales each layer's activations to have
  stable mean/variance, which speeds up and stabilizes training on a dataset this noisy.
- **Data augmentation** (`RandomFlip`, `RandomRotation`, `RandomZoom`) baked into the model as
  its first layers. It randomly perturbs each training image a little differently every epoch,
  which acts like *free extra training data* -- important here because the smallest class
  (`disgust`) has only 436 real examples to learn from. These layers are automatically no-ops
  during evaluation/prediction, so validation and test images pass through unchanged.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
], name="augmentation")

model = keras.Sequential([
    layers.Input(shape=(48, 48, 1)),
    data_augmentation,
    layers.Rescaling(1.0 / 255),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(len(class_names), activation="softmax"),
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 4. Class weights

`class_weight="balanced"` tells the loss function to penalize mistakes on rare classes
(`disgust`) more heavily than mistakes on common ones (`happy`), roughly in inverse proportion
to how many training examples each class has. Without this, the model could reach ~65-70%
accuracy by mostly ignoring `disgust` entirely, since it's only ~1.5% of the training data.

In [ ]:
train_labels = []
for i, c in enumerate(class_names):
    train_labels += [i] * class_counts[c]

class_weights = compute_class_weight(
    class_weight="balanced", classes=np.arange(len(class_names)), y=train_labels
)
class_weight_dict = dict(enumerate(class_weights))
print({class_names[i]: round(w, 2) for i, w in class_weight_dict.items()})

## 5. Train

Same `EarlyStopping` idea as before (stop once `val_loss` stalls, restore the best epoch's
weights), plus `ReduceLROnPlateau`, which halves the learning rate when validation loss plateaus
for 2 epochs -- useful here because training runs longer (harder task) and a fixed learning rate
tends to stall out partway through.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=[early_stop, reduce_lr],
    verbose=1,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Evaluate on the held-out test set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss:     {test_loss:.4f}")
print("\nFor context: human inter-annotator agreement on FER2013 labels is ~65-70%, and "
      "published benchmark CNNs (much deeper, longer-trained) top out around 72-75% test "
      "accuracy. This is genuinely one of the harder standard image datasets.")

In [ ]:
y_true = np.concatenate([y for _, y in test_ds], axis=0)
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix -- FER2013 test set")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

**What to look for in the confusion matrix**: `disgust` is almost always the weakest class
(fewest training examples), and `fear` vs `sad` / `angry` vs `disgust` are the most commonly
confused pairs -- they genuinely look alike at 48x48 resolution, even to people.

## 7. Look at real predictions

In [ ]:
for images, labels_batch in test_ds.take(1):
    preds = model.predict(images, verbose=0)
    pred_labels = np.argmax(preds, axis=1)

    plt.figure(figsize=(12, 8))
    for i in range(12):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8").squeeze(), cmap="gray")
        true_c = class_names[labels_batch[i]]
        pred_c = class_names[pred_labels[i]]
        color = "green" if true_c == pred_c else "red"
        plt.title(f"true: {true_c}\npred: {pred_c}", color=color, fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

## 8. Results & discussion

*(Fill this in once training finishes -- test accuracy, which class was weakest, and whether
class weighting / augmentation clearly helped.)*

## What could improve this further
- **Transfer learning**: start from a network pretrained on a large face dataset instead of
  training from scratch -- exactly the topic that follows the Day 3 CNN notebook.
- **More/better augmentation** or oversampling the rare classes (`disgust`) directly.
- **A deeper or residual architecture** -- FER2013 leaderboard models are usually much deeper
  than this notebook's 3-block CNN.